In [7]:
import os
import shutil
import random
import subprocess

print("1. Extracting raw dataset...")
# Using -o to safely overwrite the previous extraction
!unzip -o -q Custom_Traffic_Dataset.zip -d raw_data

# The Ultimate Dynamic Folder Finder
raw_imgs = None
raw_lbls = None

for root, dirs, files in os.walk("raw_data"):
    if "images" in dirs and "labels" in dirs:
        raw_imgs = os.path.join(root, "images")
        raw_lbls = os.path.join(root, "labels")
        print(f"-> Found your dataset perfectly at: {root}")
        break

if not raw_imgs or not raw_lbls:
    raise FileNotFoundError("Still could not find the 'images' and 'labels' folders inside the zip.")

# Create final YOLO structure (05WT1#)
base_dir = "yolo_dataset"
for split in ['train', 'val']:
    os.makedirs(f"{base_dir}/{split}/images", exist_ok=True)
    os.makedirs(f"{base_dir}/{split}/labels", exist_ok=True)

images = [f for f in os.listdir(raw_imgs) if f.endswith(('.jpg', '.png', '.jpeg'))]
random.shuffle(images)

# 80% Train, 20% Val split
split_idx = int(len(images) * 0.8)
train_imgs = images[:split_idx]
val_imgs = images[split_idx:]

def process_set(img_list, split_name):
    txt_file_path = f"{base_dir}/{split_name}.txt"
    with open(txt_file_path, 'w') as f_out:
        for img_name in img_list:
            in_img = os.path.join(raw_imgs, img_name)
            out_img = os.path.join(base_dir, split_name, "images", img_name)

            # Use ffmpeg to scale to 384 width, preserving aspect ratio (05WT2#)
            subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", in_img, "-vf", "scale=384:-1", out_img])

            # Copy corresponding label
            lbl_name = img_name.rsplit('.', 1)[0] + ".txt"
            in_lbl = os.path.join(raw_lbls, lbl_name)
            out_lbl = os.path.join(base_dir, split_name, "labels", lbl_name)
            if os.path.exists(in_lbl):
                shutil.copy(in_lbl, out_lbl)

            # Write to train.txt or val.txt paths
            f_out.write(f"./{split_name}/images/{img_name}\n")

print("2. Resizing images with ffmpeg and splitting data...")
process_set(train_imgs, 'train')
process_set(val_imgs, 'val')

# Generate the data.yaml file
yaml_content = """
train: ./train/images
val: ./val/images

nc: 2
names: ['car', 'two_wheeler']
"""
with open(f"{base_dir}/data.yaml", 'w') as f:
    f.write(yaml_content.strip())

print(f"✅ WT1 and WT2 Complete! Dataset prepped with {len(train_imgs)} train images and {len(val_imgs)} validation images.")

1. Extracting raw dataset...
-> Found your dataset perfectly at: raw_data/Week4_Task2
2. Resizing images with ffmpeg and splitting data...
✅ WT1 and WT2 Complete! Dataset prepped with 80 train images and 20 validation images.


In [8]:
# Install the Ultralytics YOLO package
!pip install -q ultralytics
from ultralytics import YOLO

print("Initializing YOLOv8 Nano base model...")
model = YOLO('yolov8n.pt')

print("Igniting Training Sequence...")
# Training for 50 epochs on your custom yaml file
results = model.train(data='/content/yolo_dataset/data.yaml', epochs=50, imgsz=384)

print("✅ WT3 Complete! Your custom weights are saved in 'runs/detect/train/weights/best.pt'")

Initializing YOLOv8 Nano base model...
Igniting Training Sequence...
Ultralytics 8.4.54 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=384, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, ops

In [9]:
import os
import glob
import subprocess
from google.colab import files

print("1. Slicing a 30-second test clip from your 4-minute video...")
# Updated to look specifically for 'test.mp4'
subprocess.run([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
    "-ss", "00:01:00", "-i", "test.mp4", "-t", "30",
    "-c", "copy", "test_30s.mp4"
])

print("2. Unleashing your custom YOLO AI on the test clip...")
# Runs inference using the exact weights you just trained
!yolo task=detect mode=predict model=runs/detect/train-2/weights/best.pt source=test_30s.mp4 save=True

print("3. Locating and Downloading the Final Result...")
# YOLO creates a new 'predict' folder for every run, this finds the newest one
predict_dirs = glob.glob('runs/detect/predict*')
if predict_dirs:
    latest_predict_dir = max(predict_dirs, key=os.path.getmtime)
    video_files = [f for f in os.listdir(latest_predict_dir) if f.endswith(('.avi', '.mp4'))]

    if video_files:
        final_video = os.path.join(latest_predict_dir, video_files[0])
        print(f"Downloading {final_video}... Check your browser downloads!")
        files.download(final_video)
    else:
        print("Error: AI processed the video but could not save the output file.")
else:
    print("Error: Prediction directory was not found.")

1. Slicing a 30-second test clip from your 4-minute video...
2. Unleashing your custom YOLO AI on the test clip...
Ultralytics 8.4.54 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs

video 1/1 (frame 1/989) /content/test_30s.mp4: 224x384 (no detections), 53.2ms
video 1/1 (frame 2/989) /content/test_30s.mp4: 224x384 (no detections), 8.2ms
video 1/1 (frame 3/989) /content/test_30s.mp4: 224x384 (no detections), 12.2ms
video 1/1 (frame 4/989) /content/test_30s.mp4: 224x384 (no detections), 8.4ms
video 1/1 (frame 5/989) /content/test_30s.mp4: 224x384 (no detections), 8.8ms
video 1/1 (frame 6/989) /content/test_30s.mp4: 224x384 (no detections), 7.5ms
video 1/1 (frame 7/989) /content/test_30s.mp4: 224x384 (no detections), 9.1ms
video 1/1 (frame 8/989) /content/test_30s.mp4: 224x384 (no detections), 7.7ms
video 1/1 (frame 9/989) /content/test_30s.mp4: 224x384 (no detections), 7.6ms
video 1/1 (frame 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import os
import glob
from google.colab import files

print("1. Forcing YOLO to show low-confidence predictions...")
# We added conf=0.05 so the AI prints boxes even if it is only 5% sure
!yolo task=detect mode=predict model=runs/detect/train-2/weights/best.pt source=test_30s.mp4 save=True conf=0.05

print("2. Locating and Downloading the Final Result...")
# YOLO creates a new 'predict' folder for every run, this finds the newest one
predict_dirs = glob.glob('runs/detect/predict*')
if predict_dirs:
    latest_predict_dir = max(predict_dirs, key=os.path.getmtime)
    video_files = [f for f in os.listdir(latest_predict_dir) if f.endswith(('.avi', '.mp4'))]

    if video_files:
        final_video = os.path.join(latest_predict_dir, video_files[0])
        print(f"Downloading {final_video}... Check your browser downloads!")
        files.download(final_video)
    else:
        print("Error: AI processed the video but could not save the output file.")
else:
    print("Error: Prediction directory was not found.")

1. Forcing YOLO to show low-confidence predictions...
Ultralytics 8.4.54 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs

video 1/1 (frame 1/989) /content/test_30s.mp4: 224x384 (no detections), 52.3ms
video 1/1 (frame 2/989) /content/test_30s.mp4: 224x384 (no detections), 8.4ms
video 1/1 (frame 3/989) /content/test_30s.mp4: 224x384 1 car, 7.3ms
video 1/1 (frame 4/989) /content/test_30s.mp4: 224x384 (no detections), 7.7ms
video 1/1 (frame 5/989) /content/test_30s.mp4: 224x384 (no detections), 8.8ms
video 1/1 (frame 6/989) /content/test_30s.mp4: 224x384 (no detections), 7.0ms
video 1/1 (frame 7/989) /content/test_30s.mp4: 224x384 (no detections), 9.0ms
video 1/1 (frame 8/989) /content/test_30s.mp4: 224x384 (no detections), 7.2ms
video 1/1 (frame 9/989) /content/test_30s.mp4: 224x384 (no detections), 8.5ms
video 1/1 (frame 10/989) /content/test_30s.mp4: 224x384 (no detections), 8.5ms
video 1/1 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>